# Full sequential transfer-learning pipeline: YOLOv5x aircraft detection

This integrated Colab notebook implements the complete course-project workflow in one file:

```text
COCO-pretrained YOLOv5x
-> HRPlanesv2 training/validation
-> HRPlanesv2 held-out test
-> CORS-ADD-HBB training/internal validation
-> official CORS val2017 held-out final test
-> qualitative inference on independently sourced Houston imagery
```

All persistent checkpoints, evaluations, manifests, and visualizations are written to `MyDrive/RS_PRO/artifacts`. Temporary prepared data under `/content` is rebuilt automatically after a Colab runtime reset. Interrupted Stage-1 or Stage-2 training resumes from the corresponding `last.pt` saved in Drive.


## Before running

1. In Colab choose a GPU runtime.
2. Confirm the project is located at `MyDrive/RS_PRO`.
3. Review the settings below. Leave `RUN_FULL_PIPELINE = True` to run the entire workflow from top to bottom.
4. Use **Runtime -> Run all**.

If GPU memory is insufficient, change `BATCH_SIZE_OVERRIDE` from `4` to `2` or `1` and run again. Completed stages are skipped, while an interrupted training stage resumes from `last.pt`.


In [ ]:
from google.colab import drive

drive.mount('/content/drive')

# Master controls for the integrated notebook.
RUN_FULL_PIPELINE = True
AUTO_RESUME = True
BATCH_SIZE_OVERRIDE = 4


# Part 1 - Data quality control and reproducible splits


## Expected Google Drive layout

```text
MyDrive/
└── RS_PRO/
    ├── HRPlanesv2/
    │   ├── train/
    │   ├── val/
    │   └── test/
    ├── CORS-ADD-HBB/
    │   ├── images/train2017/
    │   ├── images/val2017/
    │   ├── labels/train2017/
    │   └── labels/val2017/
    └── *.ipynb
```

The optional `CORS-ADD-OBB` folder is ignored by this workflow.


In [ ]:
from pathlib import Path
import csv
import json
import math
import os
import random
import re
import shutil
from collections import Counter, defaultdict

import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image, ImageDraw

PROJECT_ROOT = Path('/content/drive/MyDrive/RS_PRO')
WORK_ROOT = Path('/content/rs_pro_work')
ARTIFACT_ROOT = PROJECT_ROOT / 'artifacts'

RANDOM_SEED = 42
CORS_INTERNAL_VAL_FRACTION = 0.15
STAGE_MODE = 'copy'
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.tif', '.tiff', '.bmp'}

assert PROJECT_ROOT.exists(), f'Project folder not found: {PROJECT_ROOT}'
assert str(WORK_ROOT).startswith('/content/'), 'WORK_ROOT must remain under /content.'

required_prepared_files = [
    WORK_ROOT / 'configs' / 'hrplanesv2.yaml',
    WORK_ROOT / 'configs' / 'cors_add_hbb.yaml',
]
REBUILD_PREPARED = not all(path.exists() for path in required_prepared_files)

if REBUILD_PREPARED and WORK_ROOT.exists():
    shutil.rmtree(WORK_ROOT)

WORK_ROOT.mkdir(parents=True, exist_ok=True)
(ARTIFACT_ROOT / 'splits').mkdir(parents=True, exist_ok=True)
(ARTIFACT_ROOT / 'data_qa').mkdir(parents=True, exist_ok=True)

print('PROJECT_ROOT:', PROJECT_ROOT)
print('WORK_ROOT:', WORK_ROOT)
print('STAGE_MODE:', STAGE_MODE)
print('Rebuilding temporary prepared data:', REBUILD_PREPARED)


## Verify the source folders


In [ ]:
required_paths = [
    PROJECT_ROOT / 'HRPlanesv2' / 'train',
    PROJECT_ROOT / 'HRPlanesv2' / 'val',
    PROJECT_ROOT / 'HRPlanesv2' / 'test',
    PROJECT_ROOT / 'CORS-ADD-HBB' / 'images' / 'train2017',
    PROJECT_ROOT / 'CORS-ADD-HBB' / 'images' / 'val2017',
    PROJECT_ROOT / 'CORS-ADD-HBB' / 'labels' / 'train2017',
    PROJECT_ROOT / 'CORS-ADD-HBB' / 'labels' / 'val2017',
]

missing = [str(path) for path in required_paths if not path.exists()]
if missing:
    raise FileNotFoundError('Missing required paths:\n' + '\n'.join(missing))

for path in required_paths:
    print('OK:', path.relative_to(PROJECT_ROOT))


## Helper functions

CORS-ADD tiles are grouped by the airport/source name encoded before `_640_x_y`. The internal validation split is selected by source group rather than by individual tiles, reducing leakage from neighboring patches.


In [ ]:
def list_images(folder: Path) -> list[Path]:
    return sorted(
        path for path in folder.iterdir()
        if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
    )


def cors_source_group(image_path: Path) -> str:
    match = re.match(r'^(.*)_640_\d+_\d+$', image_path.stem)
    return match.group(1) if match else image_path.stem


def select_group_validation_split(
    images: list[Path],
    fraction: float,
    seed: int,
    trials_per_group_count: int = 2000,
) -> tuple[list[Path], list[Path], set[str]]:
    groups = defaultdict(list)
    for image in images:
        groups[cors_source_group(image)].append(image)

    group_names = sorted(groups)
    target_images = round(len(images) * fraction)
    target_group_count = max(1, round(len(group_names) * fraction))
    rng = random.Random(seed)

    best_groups = None
    best_score = None

    lower = max(1, target_group_count - 3)
    upper = min(len(group_names) - 1, target_group_count + 3)

    for group_count in range(lower, upper + 1):
        for _ in range(trials_per_group_count):
            candidate = set(rng.sample(group_names, group_count))
            candidate_count = sum(len(groups[name]) for name in candidate)
            score = (
                abs(candidate_count - target_images),
                abs(group_count - target_group_count),
                sorted(candidate),
            )
            if best_score is None or score < best_score:
                best_score = score
                best_groups = candidate

    validation_images = [
        image for image in images if cors_source_group(image) in best_groups
    ]
    training_images = [
        image for image in images if cors_source_group(image) not in best_groups
    ]

    return sorted(training_images), sorted(validation_images), set(best_groups)


def stage_image(source: Path, destination: Path, mode: str) -> None:
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists() or destination.is_symlink():
        return

    if mode == 'copy':
        shutil.copy2(source, destination)
    elif mode == 'symlink':
        os.symlink(source, destination)
    else:
        raise ValueError("STAGE_MODE must be 'copy' or 'symlink'.")


def parse_and_clean_label(source: Path, destination: Path) -> list[dict]:
    destination.parent.mkdir(parents=True, exist_ok=True)
    retained = []
    removed = []

    if not source.exists():
        removed.append({
            'file': str(source),
            'line': None,
            'reason': 'missing_label_file',
            'content': '',
        })
        destination.write_text('', encoding='utf-8')
        return removed

    for line_number, raw_line in enumerate(
        source.read_text(encoding='utf-8', errors='replace').splitlines(), start=1
    ):
        line = raw_line.strip()
        if not line:
            continue

        parts = line.split()
        reason = None

        if len(parts) != 5:
            reason = 'expected_5_columns'
        else:
            try:
                class_id = int(float(parts[0]))
                x_center, y_center, width, height = map(float, parts[1:])
                values = [x_center, y_center, width, height]

                if class_id != 0:
                    reason = 'unexpected_class_id'
                elif not all(math.isfinite(value) for value in values):
                    reason = 'non_finite_coordinate'
                elif not (0 <= x_center <= 1 and 0 <= y_center <= 1):
                    reason = 'center_outside_normalized_range'
                elif not (0 < width <= 1 and 0 < height <= 1):
                    reason = 'zero_or_invalid_box_size'
                elif (
                    x_center - width / 2 < -1e-6
                    or x_center + width / 2 > 1 + 1e-6
                    or y_center - height / 2 < -1e-6
                    or y_center + height / 2 > 1 + 1e-6
                ):
                    reason = 'box_extends_outside_image'
            except ValueError:
                reason = 'non_numeric_label'

        if reason is None:
            retained.append(line)
        else:
            removed.append({
                'file': str(source),
                'line': line_number,
                'reason': reason,
                'content': line,
            })

    destination.write_text(
        ('\n'.join(retained) + '\n') if retained else '',
        encoding='utf-8',
    )
    return removed


def stage_yolo_split(
    items: list[tuple[Path, Path, str]],
    dataset_root: Path,
    split_name: str,
    stage_mode: str,
) -> tuple[list[dict], list[dict]]:
    manifest_rows = []
    removed_rows = []

    for image_source, label_source, source_group in items:
        image_destination = dataset_root / 'images' / split_name / image_source.name
        label_destination = dataset_root / 'labels' / split_name / f'{image_source.stem}.txt'

        stage_image(image_source, image_destination, stage_mode)
        removed_rows.extend(parse_and_clean_label(label_source, label_destination))

        manifest_rows.append({
            'split': split_name,
            'source_group': source_group,
            'source_image': str(image_source),
            'source_label': str(label_source),
            'prepared_image': str(image_destination),
            'prepared_label': str(label_destination),
        })

    return manifest_rows, removed_rows


def count_label_boxes(label_folder: Path) -> int:
    count = 0
    for label_path in label_folder.glob('*.txt'):
        count += sum(
            1 for line in label_path.read_text(encoding='utf-8').splitlines()
            if line.strip()
        )
    return count


## Prepare HRPlanesv2 without changing its official split


In [ ]:
hr_source = PROJECT_ROOT / 'HRPlanesv2'
hr_prepared = WORK_ROOT / 'hrplanesv2'

hr_manifest = []
removed_annotations = []

for split_name in ['train', 'val', 'test']:
    source_folder = hr_source / split_name
    images = list_images(source_folder)
    items = [
        (image, source_folder / f'{image.stem}.txt', f'hrplanesv2_{split_name}')
        for image in images
    ]
    manifest_rows, removed_rows = stage_yolo_split(
        items, hr_prepared, split_name, STAGE_MODE
    )
    hr_manifest.extend(manifest_rows)
    removed_annotations.extend(removed_rows)
    print(f'HRPlanesv2 {split_name}: {len(images):,} images')


## Create the group-aware CORS-ADD-HBB internal validation split


In [ ]:
cors_source = PROJECT_ROOT / 'CORS-ADD-HBB'
cors_prepared = WORK_ROOT / 'cors_add_hbb'

official_train_images = list_images(cors_source / 'images' / 'train2017')
official_test_images = list_images(cors_source / 'images' / 'val2017')

cors_train_images, cors_val_images, cors_val_groups = select_group_validation_split(
    official_train_images,
    fraction=CORS_INTERNAL_VAL_FRACTION,
    seed=RANDOM_SEED,
)

train_groups = {cors_source_group(image) for image in cors_train_images}
validation_groups = {cors_source_group(image) for image in cors_val_images}

assert train_groups.isdisjoint(validation_groups)
assert len(cors_train_images) + len(cors_val_images) == len(official_train_images)

print(f'Official train2017: {len(official_train_images):,} images')
print(f'Internal training: {len(cors_train_images):,} images')
print(f'Internal validation: {len(cors_val_images):,} images')
print(f'Validation fraction of train2017: {len(cors_val_images) / len(official_train_images):.2%}')
print(f'Held-out official val2017/test: {len(official_test_images):,} images')
print(f'Train source groups: {len(train_groups)}')
print(f'Validation source groups: {len(validation_groups)}')


In [ ]:
cors_manifest = []

cors_split_sources = {
    'train': (
        cors_train_images,
        cors_source / 'labels' / 'train2017',
    ),
    'val': (
        cors_val_images,
        cors_source / 'labels' / 'train2017',
    ),
    # The folder is named val2017 in the release, but this project reserves it
    # as a held-out final test set and never uses it for checkpoint selection.
    'test': (
        official_test_images,
        cors_source / 'labels' / 'val2017',
    ),
}

for split_name, (images, label_folder) in cors_split_sources.items():
    items = [
        (image, label_folder / f'{image.stem}.txt', cors_source_group(image))
        for image in images
    ]
    manifest_rows, removed_rows = stage_yolo_split(
        items, cors_prepared, split_name, STAGE_MODE
    )
    cors_manifest.extend(manifest_rows)
    removed_annotations.extend(removed_rows)
    print(f'CORS-ADD-HBB {split_name}: {len(images):,} images')


## Save reproducible split manifests and the data-cleaning report


In [ ]:
split_root = ARTIFACT_ROOT / 'splits'
qa_root = ARTIFACT_ROOT / 'data_qa'

hr_manifest_path = split_root / 'hrplanesv2_manifest.csv'
cors_manifest_path = split_root / 'cors_add_hbb_manifest.csv'
cors_groups_path = split_root / 'cors_internal_validation_groups.json'
removed_path = qa_root / 'removed_annotations.csv'

pd.DataFrame(hr_manifest).to_csv(hr_manifest_path, index=False)
pd.DataFrame(cors_manifest).to_csv(cors_manifest_path, index=False)
pd.DataFrame(
    removed_annotations,
    columns=['file', 'line', 'reason', 'content'],
).to_csv(removed_path, index=False)

cors_groups_path.write_text(
    json.dumps(
        {
            'random_seed': RANDOM_SEED,
            'target_validation_fraction': CORS_INTERNAL_VAL_FRACTION,
            'validation_groups': sorted(cors_val_groups),
        },
        indent=2,
    ),
    encoding='utf-8',
)

print('Saved:', hr_manifest_path)
print('Saved:', cors_manifest_path)
print('Saved:', cors_groups_path)
print('Saved:', removed_path)
print('Removed invalid annotations:', len(removed_annotations))


## Create YOLOv5 dataset YAML files


In [ ]:
config_root = WORK_ROOT / 'configs'
config_root.mkdir(parents=True, exist_ok=True)

hr_yaml = config_root / 'hrplanesv2.yaml'
cors_yaml = config_root / 'cors_add_hbb.yaml'

hr_yaml.write_text(
    f"path: {hr_prepared}\n"
    "train: images/train\n"
    "val: images/val\n"
    "test: images/test\n"
    "nc: 1\n"
    "names: ['aircraft']\n",
    encoding='utf-8',
)

cors_yaml.write_text(
    f"path: {cors_prepared}\n"
    "train: images/train\n"
    "val: images/val\n"
    "test: images/test\n"
    "nc: 1\n"
    "names: ['aircraft']\n",
    encoding='utf-8',
)

# Save archival copies in Drive. Training uses the /content versions above.
shutil.copy2(hr_yaml, ARTIFACT_ROOT / 'splits' / hr_yaml.name)
shutil.copy2(cors_yaml, ARTIFACT_ROOT / 'splits' / cors_yaml.name)

print(hr_yaml.read_text())
print(cors_yaml.read_text())


## Final prepared-data summary


In [ ]:
summary_rows = []
for dataset_name, dataset_root in [
    ('HRPlanesv2', hr_prepared),
    ('CORS-ADD-HBB', cors_prepared),
]:
    for split_name in ['train', 'val', 'test']:
        image_count = len(list_images(dataset_root / 'images' / split_name))
        box_count = count_label_boxes(dataset_root / 'labels' / split_name)
        summary_rows.append({
            'dataset': dataset_name,
            'split': split_name,
            'images': image_count,
            'valid_boxes': box_count,
        })

summary = pd.DataFrame(summary_rows)
display(summary)
summary.to_csv(ARTIFACT_ROOT / 'data_qa' / 'prepared_dataset_summary.csv', index=False)


## Visual label check


In [ ]:
def draw_yolo_boxes(image_path: Path, label_path: Path, max_boxes: int = 100) -> Image.Image:
    image = Image.open(image_path).convert('RGB')
    draw = ImageDraw.Draw(image)
    width, height = image.size

    lines = label_path.read_text(encoding='utf-8').splitlines()
    for line in lines[:max_boxes]:
        class_id, x_center, y_center, box_width, box_height = map(float, line.split())
        x1 = (x_center - box_width / 2) * width
        y1 = (y_center - box_height / 2) * height
        x2 = (x_center + box_width / 2) * width
        y2 = (y_center + box_height / 2) * height
        draw.rectangle((x1, y1, x2, y2), outline='red', width=max(2, width // 1000))

    return image


samples = [
    ('HRPlanesv2 train', hr_prepared / 'images' / 'train'),
    ('CORS-ADD-HBB train', cors_prepared / 'images' / 'train'),
    ('CORS-ADD-HBB internal val', cors_prepared / 'images' / 'val'),
    ('CORS-ADD-HBB held-out test', cors_prepared / 'images' / 'test'),
]

figure, axes = plt.subplots(2, 2, figsize=(14, 10))
for axis, (title, image_folder) in zip(axes.ravel(), samples):
    image_path = list_images(image_folder)[0]
    label_path = image_folder.parent.parent / 'labels' / image_folder.name / f'{image_path.stem}.txt'
    checked_image = draw_yolo_boxes(image_path, label_path)
    axis.imshow(checked_image)
    axis.set_title(title)
    axis.axis('off')

plt.tight_layout()
plt.show()


## Next step

Keep this Colab runtime active and open `01_Sequential_Transfer_Learning_YOLOv5x.ipynb`. The next notebook expects these files:

- `/content/rs_pro_work/configs/hrplanesv2.yaml`
- `/content/rs_pro_work/configs/cors_add_hbb.yaml`

If the runtime resets, rerun this notebook before resuming training.


# Part 2 - Sequential YOLOv5x transfer learning and evaluation


In [ ]:
from pathlib import Path
import json
import os
import platform
import shutil
import subprocess
import sys

import pandas as pd
import torch

PROJECT_ROOT = Path('/content/drive/MyDrive/RS_PRO')
WORK_ROOT = Path('/content/rs_pro_work')
ARTIFACT_ROOT = PROJECT_ROOT / 'artifacts'
RUNS_ROOT = ARTIFACT_ROOT / 'runs'
EVALUATION_ROOT = ARTIFACT_ROOT / 'evaluation'

YOLOV5_ROOT = Path('/content/yolov5')
YOLOV5_REF = 'master'

HR_YAML = WORK_ROOT / 'configs' / 'hrplanesv2.yaml'
CORS_YAML = WORK_ROOT / 'configs' / 'cors_add_hbb.yaml'

IMAGE_SIZE = 640
BATCH_SIZE = 4
EPOCHS_STAGE1 = 100
EPOCHS_STAGE2 = 100
PATIENCE = 20
WORKERS = 2
RANDOM_SEED = 42

RUNS_ROOT.mkdir(parents=True, exist_ok=True)
EVALUATION_ROOT.mkdir(parents=True, exist_ok=True)

for required in [PROJECT_ROOT, HR_YAML, CORS_YAML]:
    if not required.exists():
        raise FileNotFoundError(
            f'Missing {required}. Run notebook 00 in this Colab runtime first.'
        )

print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
print('PyTorch:', torch.__version__)
print('Python:', sys.version)


In [ ]:
BATCH_SIZE = BATCH_SIZE_OVERRIDE
print('Effective training/evaluation batch size:', BATCH_SIZE)


## Install the official YOLOv5 implementation


In [ ]:
from collections import deque


def run_streaming(command: list[str], cwd: Path, tail_lines: int = 120) -> None:
    # Run a long command with live logs and retain a useful error tail.
    print(' '.join(map(str, command)))
    recent_output = deque(maxlen=tail_lines)
    process = subprocess.Popen(
        command,
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='')
        recent_output.append(line.rstrip())
    return_code = process.wait()
    if return_code != 0:
        error_tail = '\n'.join(recent_output)
        raise RuntimeError(
            f'Command failed with exit code {return_code}. '
            f'The final {len(recent_output)} log lines were:\n{error_tail}'
        )


# A previous attempt may have left the old v7.0 tag in /content.
# Replace it automatically with the maintained official branch.
if YOLOV5_ROOT.exists():
    current_branch = subprocess.run(
        ['git', '-C', str(YOLOV5_ROOT), 'rev-parse', '--abbrev-ref', 'HEAD'],
        capture_output=True,
        text=True,
        check=False,
    ).stdout.strip()
    if current_branch != YOLOV5_REF:
        print(
            f'Removing incompatible YOLOv5 checkout '
            f'(current={current_branch or "unknown"}, required={YOLOV5_REF}).'
        )
        shutil.rmtree(YOLOV5_ROOT)

if not YOLOV5_ROOT.exists():
    subprocess.run(
        [
            'git', 'clone', '--depth', '1', '--branch', YOLOV5_REF,
            'https://github.com/ultralytics/yolov5.git',
            str(YOLOV5_ROOT),
        ],
        check=True,
    )

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-r', str(YOLOV5_ROOT / 'requirements.txt')],
    check=True,
)

commit = subprocess.run(
    ['git', '-C', str(YOLOV5_ROOT), 'rev-parse', 'HEAD'],
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()

environment = {
    'yolov5_ref': YOLOV5_REF,
    'yolov5_commit': commit,
    'python': sys.version,
    'pytorch': torch.__version__,
    'cuda_available': torch.cuda.is_available(),
    'cuda_version': torch.version.cuda,
    'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    'image_size': IMAGE_SIZE,
    'batch_size': BATCH_SIZE,
    'random_seed': RANDOM_SEED,
}

(ARTIFACT_ROOT / 'environment.json').write_text(
    json.dumps(environment, indent=2), encoding='utf-8'
)
print(json.dumps(environment, indent=2))
print('YOLOv5 compatibility checkout is ready:', commit)


## Stage 1 - Fine-tune COCO-pretrained YOLOv5x on HRPlanesv2

`train.py` evaluates the HRPlanesv2 validation split after every epoch and writes `best.pt` based on validation fitness. The test split is not used by this command.


In [ ]:
stage1_name = 'stage1_hrplanesv2_yolov5x'
stage1_dir = RUNS_ROOT / stage1_name
HR_BEST = stage1_dir / 'weights' / 'best.pt'
HR_LAST = stage1_dir / 'weights' / 'last.pt'
STAGE1_COMPLETE = stage1_dir / 'stage_complete.txt'

stage1_command = [
    sys.executable,
    str(YOLOV5_ROOT / 'train.py'),
    '--weights', 'yolov5x.pt',
    '--data', str(HR_YAML),
    '--img', str(IMAGE_SIZE),
    '--batch-size', str(BATCH_SIZE),
    '--epochs', str(EPOCHS_STAGE1),
    '--patience', str(PATIENCE),
    '--workers', str(WORKERS),
    '--seed', str(RANDOM_SEED),
    '--save-period', '10',
    '--project', str(RUNS_ROOT),
    '--name', stage1_name,
    '--exist-ok',
]

if RUN_FULL_PIPELINE:
    if STAGE1_COMPLETE.exists() and HR_BEST.exists():
        print('Stage 1 is already complete; skipping:', HR_BEST)
    else:
        if AUTO_RESUME and HR_LAST.exists():
            command = [
                sys.executable,
                str(YOLOV5_ROOT / 'train.py'),
                '--resume', str(HR_LAST),
            ]
            print('Resuming Stage 1 from:', HR_LAST)
        else:
            command = stage1_command
            print('Starting Stage 1 from COCO-pretrained yolov5x.pt')

        run_streaming(command, cwd=YOLOV5_ROOT)
        if not HR_BEST.exists():
            raise FileNotFoundError(f'Stage-1 best.pt was not produced: {HR_BEST}')
        STAGE1_COMPLETE.write_text('complete\n', encoding='utf-8')
        print('Stage 1 complete:', HR_BEST)
else:
    print('Pipeline execution is disabled. Set RUN_FULL_PIPELINE = True near the top.')


## Inspect Stage-1 training history


In [ ]:
stage1_results = stage1_dir / 'results.csv'
if stage1_results.exists():
    stage1_history = pd.read_csv(stage1_results)
    display(stage1_history.tail())
else:
    print('No Stage-1 results yet:', stage1_results)


## Evaluate the selected Stage-1 model on the HRPlanesv2 test split

Run this only after Stage 1 has completed. The resulting metrics describe the HRPlanesv2 stage and must not be used to select the checkpoint.


In [ ]:
hr_test_name = 'hrplanesv2_stage1_test'
hr_test_dir = EVALUATION_ROOT / hr_test_name
HR_TEST_COMPLETE = hr_test_dir / 'evaluation_complete.txt'

hr_test_command = [
    sys.executable,
    str(YOLOV5_ROOT / 'val.py'),
    '--weights', str(HR_BEST),
    '--data', str(HR_YAML),
    '--task', 'test',
    '--img', str(IMAGE_SIZE),
    '--batch-size', str(BATCH_SIZE),
    '--workers', str(WORKERS),
    '--save-txt',
    '--save-conf',
    '--save-json',
    '--project', str(EVALUATION_ROOT),
    '--name', hr_test_name,
    '--exist-ok',
]

if RUN_FULL_PIPELINE:
    if HR_TEST_COMPLETE.exists():
        print('HRPlanesv2 held-out test is already complete; skipping.')
    else:
        if not HR_BEST.exists():
            raise FileNotFoundError(f'Stage-1 best.pt not found: {HR_BEST}')
        run_streaming(hr_test_command, cwd=YOLOV5_ROOT)
        hr_test_dir.mkdir(parents=True, exist_ok=True)
        HR_TEST_COMPLETE.write_text('complete\n', encoding='utf-8')
        print('HRPlanesv2 held-out test complete:', hr_test_dir)


## Stage 2 - Fine-tune the Stage-1 model on CORS-ADD-HBB

This stage must initialize from the HRPlanesv2 `best.pt`, not from the original COCO weight. The internal CORS validation split selects the final checkpoint. The official `val2017` images remain untouched during training.


In [ ]:
stage2_name = 'stage2_cors_add_hbb_yolov5x'
stage2_dir = RUNS_ROOT / stage2_name
CORS_BEST = stage2_dir / 'weights' / 'best.pt'
CORS_LAST = stage2_dir / 'weights' / 'last.pt'
STAGE2_COMPLETE = stage2_dir / 'stage_complete.txt'

stage2_command = [
    sys.executable,
    str(YOLOV5_ROOT / 'train.py'),
    '--weights', str(HR_BEST),
    '--data', str(CORS_YAML),
    '--img', str(IMAGE_SIZE),
    '--batch-size', str(BATCH_SIZE),
    '--epochs', str(EPOCHS_STAGE2),
    '--patience', str(PATIENCE),
    '--workers', str(WORKERS),
    '--seed', str(RANDOM_SEED),
    '--save-period', '10',
    '--project', str(RUNS_ROOT),
    '--name', stage2_name,
    '--exist-ok',
]

if RUN_FULL_PIPELINE:
    if STAGE2_COMPLETE.exists() and CORS_BEST.exists():
        print('Stage 2 is already complete; skipping:', CORS_BEST)
    else:
        if AUTO_RESUME and CORS_LAST.exists():
            command = [
                sys.executable,
                str(YOLOV5_ROOT / 'train.py'),
                '--resume', str(CORS_LAST),
            ]
            print('Resuming Stage 2 from:', CORS_LAST)
        else:
            if not HR_BEST.exists():
                raise FileNotFoundError(f'Stage-1 best.pt not found: {HR_BEST}')
            command = stage2_command
            print('Starting Stage 2 from the Stage-1 HRPlanesv2 checkpoint')

        run_streaming(command, cwd=YOLOV5_ROOT)
        if not CORS_BEST.exists():
            raise FileNotFoundError(f'Stage-2 best.pt was not produced: {CORS_BEST}')
        STAGE2_COMPLETE.write_text('complete\n', encoding='utf-8')
        print('Stage 2 complete:', CORS_BEST)


## Inspect Stage-2 training history


In [ ]:
stage2_results = stage2_dir / 'results.csv'
if stage2_results.exists():
    stage2_history = pd.read_csv(stage2_results)
    display(stage2_history.tail())
else:
    print('No Stage-2 results yet:', stage2_results)


## Final held-out CORS-ADD-HBB test

The CORS YAML maps the original `val2017` release folder to `test`. Run this cell once after the final Stage-2 checkpoint has been selected. Do not use these results to change training settings or choose a different checkpoint.


In [ ]:
cors_test_name = 'cors_add_hbb_final_test'
cors_test_dir = EVALUATION_ROOT / cors_test_name
CORS_TEST_COMPLETE = cors_test_dir / 'evaluation_complete.txt'

cors_test_command = [
    sys.executable,
    str(YOLOV5_ROOT / 'val.py'),
    '--weights', str(CORS_BEST),
    '--data', str(CORS_YAML),
    '--task', 'test',
    '--img', str(IMAGE_SIZE),
    '--batch-size', str(BATCH_SIZE),
    '--workers', str(WORKERS),
    '--save-txt',
    '--save-conf',
    '--save-json',
    '--project', str(EVALUATION_ROOT),
    '--name', cors_test_name,
    '--exist-ok',
]

if RUN_FULL_PIPELINE:
    if CORS_TEST_COMPLETE.exists():
        print('Final held-out CORS test is already complete; skipping.')
    else:
        if not CORS_BEST.exists():
            raise FileNotFoundError(f'Final best.pt not found: {CORS_BEST}')
        run_streaming(cors_test_command, cwd=YOLOV5_ROOT)
        cors_test_dir.mkdir(parents=True, exist_ok=True)
        CORS_TEST_COMPLETE.write_text('complete\n', encoding='utf-8')
        print('Final held-out CORS test complete:', cors_test_dir)


## Outputs

Important outputs remain in Google Drive:

- `RS_PRO/artifacts/runs/stage1_hrplanesv2_yolov5x/weights/best.pt`
- `RS_PRO/artifacts/runs/stage2_cors_add_hbb_yolov5x/weights/best.pt`
- `RS_PRO/artifacts/evaluation/hrplanesv2_stage1_test/`
- `RS_PRO/artifacts/evaluation/cors_add_hbb_final_test/`

Continue with notebook 02 after the final Stage-2 model is available.


# Part 3 - Houston external qualitative inference


In [ ]:
from pathlib import Path
import csv
import json
import math
import re
import shutil
import subprocess
import sys

import matplotlib.pyplot as plt
import pandas as pd
import torch
from PIL import Image, ImageDraw, ImageFont
from torchvision.ops import nms

PROJECT_ROOT = Path('/content/drive/MyDrive/RS_PRO')
ARTIFACT_ROOT = PROJECT_ROOT / 'artifacts'
YOLOV5_ROOT = Path('/content/yolov5')

FINAL_WEIGHTS = (
    ARTIFACT_ROOT
    / 'runs'
    / 'stage2_cors_add_hbb_yolov5x'
    / 'weights'
    / 'best.pt'
)

HOUSTON_INPUT = PROJECT_ROOT / 'Houston_Imagery' / 'input'
HOUSTON_OUTPUT = ARTIFACT_ROOT / 'houston_external_inference'

LOCAL_TILE_ROOT = Path('/content/houston_tiles')
LOCAL_DETECTION_ROOT = Path('/content/houston_detection')

TILE_SIZE = 640
TILE_OVERLAP = 0.25
CONFIDENCE_THRESHOLD = 0.25
DETECTION_IOU_THRESHOLD = 0.45
GLOBAL_NMS_IOU_THRESHOLD = 0.40

HOUSTON_INPUT.mkdir(parents=True, exist_ok=True)
HOUSTON_OUTPUT.mkdir(parents=True, exist_ok=True)

print('Place independently sourced Houston images in:', HOUSTON_INPUT)
print('Final weights:', FINAL_WEIGHTS)


## Record image provenance

Before inference, create or complete `Houston_Imagery/metadata.csv` with one row per source image. Record the provider, acquisition date, spatial resolution, airport, and confirmation that the image does not duplicate or overlap CORS-ADD imagery.


In [ ]:
metadata_path = PROJECT_ROOT / 'Houston_Imagery' / 'metadata.csv'

if not metadata_path.exists():
    pd.DataFrame(
        columns=[
            'filename',
            'airport',
            'source_or_provider',
            'source_url',
            'acquisition_date',
            'spatial_resolution',
            'license_or_terms',
            'non_overlap_with_cors_add_confirmed',
            'notes',
        ]
    ).to_csv(metadata_path, index=False)
    print('Created metadata template:', metadata_path)
else:
    display(pd.read_csv(metadata_path))


## Install YOLOv5 if this is a new Colab runtime


In [ ]:
YOLOV5_REF = 'master'

if not YOLOV5_ROOT.exists():
    subprocess.run(
        [
            'git', 'clone', '--depth', '1', '--branch', YOLOV5_REF,
            'https://github.com/ultralytics/yolov5.git',
            str(YOLOV5_ROOT),
        ],
        check=True,
    )

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-r', str(YOLOV5_ROOT / 'requirements.txt')],
    check=True,
)

if not FINAL_WEIGHTS.exists():
    raise FileNotFoundError(
        f'Final Stage-2 weights are missing: {FINAL_WEIGHTS}. Run notebook 01 first.'
    )


## Tile the large Houston images


In [ ]:
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.tif', '.tiff', '.bmp'}


def slugify(value: str) -> str:
    value = re.sub(r'[^A-Za-z0-9_-]+', '_', value)
    return value.strip('_') or 'image'


def axis_positions(length: int, tile_size: int, stride: int) -> list[int]:
    if length <= tile_size:
        return [0]
    positions = list(range(0, length - tile_size + 1, stride))
    final_position = length - tile_size
    if positions[-1] != final_position:
        positions.append(final_position)
    return positions


def tile_images(
    source_paths: list[Path],
    tile_root: Path,
    tile_size: int,
    overlap: float,
) -> pd.DataFrame:
    if tile_root.exists():
        shutil.rmtree(tile_root)
    tile_root.mkdir(parents=True, exist_ok=True)

    stride = max(1, round(tile_size * (1 - overlap)))
    manifest_rows = []

    for source_index, source_path in enumerate(source_paths):
        with Image.open(source_path) as source_image:
            source_image = source_image.convert('RGB')
            width, height = source_image.size

            x_positions = axis_positions(width, tile_size, stride)
            y_positions = axis_positions(height, tile_size, stride)

            for y0 in y_positions:
                for x0 in x_positions:
                    x1 = min(x0 + tile_size, width)
                    y1 = min(y0 + tile_size, height)
                    tile = source_image.crop((x0, y0, x1, y1))

                    tile_name = (
                        f'{source_index:03d}_{slugify(source_path.stem)}'
                        f'__x{x0}__y{y0}.png'
                    )
                    tile_path = tile_root / tile_name
                    tile.save(tile_path)

                    manifest_rows.append({
                        'tile_name': tile_name,
                        'tile_path': str(tile_path),
                        'source_path': str(source_path),
                        'source_name': source_path.name,
                        'x0': x0,
                        'y0': y0,
                        'tile_width': x1 - x0,
                        'tile_height': y1 - y0,
                        'source_width': width,
                        'source_height': height,
                    })

    return pd.DataFrame(manifest_rows)


houston_images = sorted(
    path for path in HOUSTON_INPUT.iterdir()
    if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
)

if not houston_images:
    raise FileNotFoundError(
        f'No Houston images found in {HOUSTON_INPUT}. Add images before continuing.'
    )

tile_manifest = tile_images(
    houston_images,
    LOCAL_TILE_ROOT,
    tile_size=TILE_SIZE,
    overlap=TILE_OVERLAP,
)

tile_manifest_path = HOUSTON_OUTPUT / 'tile_manifest.csv'
tile_manifest.to_csv(tile_manifest_path, index=False)

print(f'Source images: {len(houston_images)}')
print(f'Tiles: {len(tile_manifest)}')
print('Manifest:', tile_manifest_path)
display(tile_manifest.head())


## Run tile-level inference


In [ ]:
RUN_HOUSTON_INFERENCE = RUN_FULL_PIPELINE

detection_command = [
    sys.executable,
    str(YOLOV5_ROOT / 'detect.py'),
    '--weights', str(FINAL_WEIGHTS),
    '--source', str(LOCAL_TILE_ROOT),
    '--img', str(TILE_SIZE),
    '--conf-thres', str(CONFIDENCE_THRESHOLD),
    '--iou-thres', str(DETECTION_IOU_THRESHOLD),
    '--save-txt',
    '--save-conf',
    '--project', str(LOCAL_DETECTION_ROOT.parent),
    '--name', LOCAL_DETECTION_ROOT.name,
    '--exist-ok',
]

print(' '.join(detection_command))
if RUN_HOUSTON_INFERENCE:
    if LOCAL_DETECTION_ROOT.exists():
        shutil.rmtree(LOCAL_DETECTION_ROOT)
    run_streaming(detection_command, cwd=YOLOV5_ROOT)
else:
    raise RuntimeError('Set RUN_FULL_PIPELINE = True near the top before Run all.')


## Map tile detections back to full images and remove duplicates


In [ ]:
def yolo_detection_to_global(
    parts: list[str],
    tile_row: pd.Series,
) -> dict:
    class_id, x_center, y_center, width, height, confidence = map(float, parts)

    tile_width = float(tile_row['tile_width'])
    tile_height = float(tile_row['tile_height'])

    x1 = (x_center - width / 2) * tile_width + float(tile_row['x0'])
    y1 = (y_center - height / 2) * tile_height + float(tile_row['y0'])
    x2 = (x_center + width / 2) * tile_width + float(tile_row['x0'])
    y2 = (y_center + height / 2) * tile_height + float(tile_row['y0'])

    return {
        'source_path': tile_row['source_path'],
        'source_name': tile_row['source_name'],
        'class_id': int(class_id),
        'confidence': confidence,
        'x1': max(0.0, x1),
        'y1': max(0.0, y1),
        'x2': min(float(tile_row['source_width']), x2),
        'y2': min(float(tile_row['source_height']), y2),
        'tile_name': tile_row['tile_name'],
    }


def collect_global_detections(
    manifest: pd.DataFrame,
    labels_root: Path,
) -> pd.DataFrame:
    columns = [
        'source_path', 'source_name', 'class_id', 'confidence',
        'x1', 'y1', 'x2', 'y2', 'tile_name',
    ]
    detections = []
    for _, row in manifest.iterrows():
        label_path = labels_root / f"{Path(row['tile_name']).stem}.txt"
        if not label_path.exists():
            continue

        for line in label_path.read_text(encoding='utf-8').splitlines():
            parts = line.split()
            if len(parts) != 6:
                continue
            detections.append(yolo_detection_to_global(parts, row))

    return pd.DataFrame(detections, columns=columns)


def apply_global_nms(
    detections: pd.DataFrame,
    iou_threshold: float,
) -> pd.DataFrame:
    if detections.empty:
        return detections.copy()

    retained_frames = []
    for source_path, frame in detections.groupby('source_path', sort=True):
        boxes = torch.tensor(
            frame[['x1', 'y1', 'x2', 'y2']].to_numpy(),
            dtype=torch.float32,
        )
        scores = torch.tensor(frame['confidence'].to_numpy(), dtype=torch.float32)
        keep_indices = nms(boxes, scores, iou_threshold).cpu().numpy()
        retained_frames.append(frame.iloc[keep_indices])

    return pd.concat(retained_frames, ignore_index=True)


labels_root = LOCAL_DETECTION_ROOT / 'labels'
if not labels_root.exists():
    raise FileNotFoundError(
        f'Detection labels not found: {labels_root}. Run the inference cell first.'
    )

raw_detections = collect_global_detections(tile_manifest, labels_root)
final_detections = apply_global_nms(
    raw_detections,
    iou_threshold=GLOBAL_NMS_IOU_THRESHOLD,
)

raw_detections.to_csv(HOUSTON_OUTPUT / 'raw_tile_detections.csv', index=False)
final_detections.to_csv(HOUSTON_OUTPUT / 'final_global_detections.csv', index=False)

print('Raw tile detections:', len(raw_detections))
print('Detections after global NMS:', len(final_detections))
display(final_detections.head())


## Save full-image detection visualizations


In [ ]:
visualization_root = HOUSTON_OUTPUT / 'visualizations'
visualization_root.mkdir(parents=True, exist_ok=True)

for source_path_string in sorted(final_detections['source_path'].unique()):
    source_path = Path(source_path_string)
    with Image.open(source_path) as image:
        image = image.convert('RGB')
        draw = ImageDraw.Draw(image)

        frame = final_detections[
            final_detections['source_path'] == source_path_string
        ].sort_values('confidence', ascending=False)

        line_width = max(2, min(image.size) // 1000)
        for _, detection in frame.iterrows():
            box = (
                detection['x1'],
                detection['y1'],
                detection['x2'],
                detection['y2'],
            )
            draw.rectangle(box, outline='red', width=line_width)
            label = f"aircraft {detection['confidence']:.2f}"
            draw.text((detection['x1'], max(0, detection['y1'] - 14)), label, fill='red')

        output_path = visualization_root / f'{source_path.stem}_detections.jpg'
        image.save(output_path, quality=95)
        print('Saved:', output_path)


## Preview results


In [ ]:
visualization_paths = sorted(visualization_root.glob('*.jpg'))

if not visualization_paths:
    print('No visualizations were produced.')
else:
    figure, axes = plt.subplots(
        len(visualization_paths),
        1,
        figsize=(16, 8 * len(visualization_paths)),
    )
    if len(visualization_paths) == 1:
        axes = [axes]

    for axis, visualization_path in zip(axes, visualization_paths):
        image = Image.open(visualization_path)
        axis.imshow(image)
        axis.set_title(visualization_path.name)
        axis.axis('off')

    plt.tight_layout()
    plt.show()


## Interpretation rule

Report these Houston results as qualitative external inference unless complete ground-truth annotations are created. Discuss representative correct detections, apparent false positives, apparent false negatives, dense parking areas, small aircraft, and background complexity. Do not report mAP, Precision, Recall, or F1 from unannotated Houston imagery.
